# Media Framing Gold Evaluation

Purpose: compare the manual `GOLD TRUTH` labels against the old GPT label from the manual annotation sheet and the new multi-model labels.

Inputs:
- Manual annotation export, copied into the project as `03_Framing/outputs/test_runs/framing_manual_gold_labels_100.csv`.
- New multi-model long labels from `03_Framing/outputs/test_runs/framing_annotation_sample_100_multimodel_labels_long.csv`.

This notebook deliberately uses only `GOLD TRUTH`, `GPT Label`, and `GPT Evidence` from the manual annotation file. Annotator-specific columns are ignored.


In [1]:
from pathlib import Path
import math
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 180)


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / '.git').exists():
            return candidate
    raise FileNotFoundError('Could not find project root containing .git')

PROJECT_ROOT = find_project_root(Path.cwd())
TEST_DIR = PROJECT_ROOT / '03_Framing' / 'outputs' / 'test_runs'

MANUAL_LABELS_PATH = TEST_DIR / 'framing_manual_gold_labels_100.csv'
MULTIMODEL_LONG_PATH = TEST_DIR / 'framing_annotation_sample_100_multimodel_labels_long.csv'

EVAL_LONG_PATH = TEST_DIR / 'framing_gold_model_evaluation_long.csv'
EVAL_WIDE_PATH = TEST_DIR / 'framing_gold_model_evaluation_wide.csv'
OUTLET_ACCURACY_PATH = TEST_DIR / 'framing_gold_accuracy_by_outlet.csv'
OUTLET_ACCURACY_PIVOT_PATH = TEST_DIR / 'framing_gold_accuracy_by_outlet_pivot.csv'
KAPPA_SUMMARY_PATH = TEST_DIR / 'framing_gold_kappa_summary.csv'
OUTLET_KAPPA_PATH = TEST_DIR / 'framing_gold_kappa_by_outlet.csv'

print('Project root:', PROJECT_ROOT)
print('Manual labels:', MANUAL_LABELS_PATH)
print('Multi-model long labels:', MULTIMODEL_LONG_PATH)

Project root: /Users/MattisHaumann/Dev/Thesis
Manual labels: /Users/MattisHaumann/Dev/Thesis/03_Framing/outputs/test_runs/framing_manual_gold_labels_100.csv
Multi-model long labels: /Users/MattisHaumann/Dev/Thesis/03_Framing/outputs/test_runs/framing_annotation_sample_100_multimodel_labels_long.csv


In [2]:
def load_first_sheet_or_csv(path: Path) -> pd.DataFrame:
    if path.suffix.lower() in {'.xlsx', '.xls'}:
        return pd.read_excel(path, sheet_name=0)
    return pd.read_csv(path)


def normalize_label(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).strip()
    if not text or text.lower() in {'nan', 'none', 'na', 'n/a'}:
        return pd.NA
    return text

manual_raw = load_first_sheet_or_csv(MANUAL_LABELS_PATH)
manual_raw = manual_raw.rename(columns={col: str(col).strip() for col in manual_raw.columns})

required_manual_cols = [
    'item_id', 'hit_id', 'row_id', 'outlet', 'article_title', 'entity_mention', 'context_window',
    'GOLD TRUTH', 'GPT Label', 'GPT Evidence'
]
missing = [col for col in required_manual_cols if col not in manual_raw.columns]
if missing:
    raise KeyError(f'Missing expected manual columns: {missing}')

manual = manual_raw.loc[:, required_manual_cols].copy()
manual = manual.rename(columns={
    'context_window': 'text',
    'GOLD TRUTH': 'gold_label',
    'GPT Label': 'old_gpt_5_mini_category',
    'GPT Evidence': 'old_gpt_5_mini_evidence',
})
manual['gold_label'] = manual['gold_label'].apply(normalize_label)
manual['old_gpt_5_mini_category'] = manual['old_gpt_5_mini_category'].apply(normalize_label)
manual['old_gpt_5_mini_evidence'] = manual['old_gpt_5_mini_evidence'].apply(lambda x: '' if pd.isna(x) else str(x).strip())

print('Manual rows:', len(manual))
print('Rows with GOLD TRUTH:', int(manual['gold_label'].notna().sum()))
print('Rows without GOLD TRUTH:', int(manual['gold_label'].isna().sum()))
display(manual.head(10))

Manual rows: 100
Rows with GOLD TRUTH: 98
Rows without GOLD TRUTH: 2


,item_id,hit_id,row_id,outlet,article_title,entity_mention,text,gold_label,old_gpt_5_mini_category,old_gpt_5_mini_evidence
0,1,5718ec8e2e0e1a02,2722,Nius,"„Rechtsextrem“, „rassistisch“, „Fake News“: Diese Lügen verbreiten deutsche Medien unmittelbar nach Kirks Ermordung über ihn",Spiegel | Tagesschau,"Das Attentat auf Charlie Kirk rührte viele Menschen wie hier in Utah – linke deutsche Medien sahen dagegen Anlass für Märchenerzählungen. Der einzige Vorwurf, den der Spiegel ü...",DISINFORMATION/FALSCHDARSTELLUNG,POSITIONS-/PARTEILICHKEITS-BIAS,linke deutsche Medien
1,2,7887b57e328ce932,4764,Nius,"Queere Tiere auf der Arche, Windel-Hühnchen in der Kirche: So verrückt war 2025",ZDF | Stern,"Merz, wieder einmal wahrheitswidrig: „Es gibt zwischen der CDU und der AfD keine Gemeinsamkeiten.“ Eine Schlagzeile, wie sie nur in Deutschland denkbar ist: „Unbekannte hissen ...",NaN,VERZERRUNG/MANIPULATION,Rechte Klassenzimmer
2,3,168f480d060f709d,4920,Nius,Der Winter hat seine Unschuld verloren: Früher bedeutete Schnee pure Lebensfreude – heute wird den Kindern Angst gemacht,Tagesschau,Zudem: Die angekündigten extremen Schneefälle ließen am Ende auf sich warten. Betreutes Laufen: Diese Anleitung postete die Tagesschau auf Instagram. Die Natur wird zusehends z...,NEUTRAL,VERZERRUNG/MANIPULATION,zu unserem Feind stilisiert
3,4,b3ebf0fb7f6da9a9,9073,RT_de,"Lanz in Panik: ""Russland hat sich nach Westen ausgeweitet""",Markus Lanz | ZDF,"Lanz in Panik: ""Russland hat sich nach Westen ausgeweitet"". Von Alexej Danckwardt Wussten Sie, dass nicht die NATO sich nach 1991 um Hunderte Kilometer nach Osten ausgedehnt ha...",VERSAGEN/INKOMPETENZ,VERSAGEN/INKOMPETENZ,weniger Gegenwehr entgegenzusetzen
4,5,df33ae235b273262,7663,RT_de,"ZDF-Mitarbeiter beklagen interne Repressionen und fehlende ""innere Pressefreiheit""",WDR | ZDF,"Tucker Carlson: Zensur schützt die Mächtigen, nicht die Schwachen Im Verlauf der Anhörung benannte Halbach zwei Beispiele für ""Einschüchterungsversuche"" gegenüber kritischen St...",VERZERRUNG/MANIPULATION,POSITIONS-/PARTEILICHKEITS-BIAS,"Zensur schützt die Mächtigen, nicht die Schwachen"
5,6,4714abd36db0c52e,3438,Nius,Gefangen im Land der Durchhalteparolen: Das hohle Stadtbild-Gedröhne des Friedrich Merz,Tagesschau,"Das Stadtbild ändert sich, ohne dass die Bevölkerung jemals gefragt wurde, ob es diese Veränderung wirklich in Kauf nehmen will. ## Synthese mit einem expansiven Islam Der vere...",NEUTRAL,POSITIONS-/PARTEILICHKEITS-BIAS,gebilligt von Tagesschau-Boomern
6,7,a8324d32b15f68db,11397,Tichys_Einblick,"Wacht auf, ihr Schlafwandler!",NDR,"Wacht auf, ihr Schlafwandler! Im literarischen Freiheitschor gegen deutsche Staatsgläubigkeit dominieren wenige markante Stimmen: So geben beispielsweise der Buchautor Gerald G...",NEUTRAL,NEUTRAL,
7,8,c81d317f74eb603d,425,Antispiegel,New York Times: „Selenskys Regierung sabotierte die Aufsicht und ließ Korruption gedeihen“,Spiegel | Qualitätsmedien,"Selensky habe damit rein gar nichts zu tun, auch wenn sein gesamtes Umfeld darin verstrickt ist und das NABU offen davon spricht, sie hätten den Schutz und die Nähe Selenskys f...",VERZERRUNG/MANIPULATION,VERZERRUNG/MANIPULATION,Spiegel-Propagandist
8,9,c948446bc708c8ff,3289,Nius,"An alle, die Trump mit Hitler verglichen haben – schämt euch!",FAZ | Spiegel | Markus Lanz,"Er trägt die Verantwortung für diese Eskalation.“ Die „Aushöhlung der Demokratie“ von rechts, die auch der AfD unterstellt wurde, treibt auch Heidi Reichinnek, die Vorsitzende ...",VERZERRUNG/MANIPULATION,POSITIONS-/PARTEILICHKEITS-BIAS,als Verkörperung des Bösen
9,10,35a3991c3a95121c,16910,Tagesschau,Neuer Nestlé-Chef Navratil streicht 16.000 Stellen,Deutschlandfunk,"Denn bei Schlüsselindikatoren wie dem Umsatzwachstum und der Aktienpreisentwicklung hinkt der Nahrungsmittelriese, dessen Produktpalette von Fertiggerichten und Tiefkühlprodukt...",NEUTRAL,NEUTRAL,


In [3]:
labels_long = pd.read_csv(MULTIMODEL_LONG_PATH)
labels_long = labels_long.loc[labels_long['run_status'].eq('ok')].copy()

# Deduplicate defensively, preferring latest ok row if a model was retried.
labels_long = (
    labels_long.sort_values(['item_id', 'prompt_variant', 'provider', 'model', 'created_at_utc'])
    .drop_duplicates(['item_id', 'prompt_variant', 'provider', 'model'], keep='last')
)

model_key_map = {
    ('openai', 'gpt-5.4'): ('new_openai_gpt_5_4', 'GPT-5.4'),
    ('openai', 'gpt-5.4-mini'): ('new_openai_gpt_5_4_mini', 'GPT-5.4 mini'),
    ('openai', 'gpt-5-mini'): ('new_openai_gpt_5_mini', 'GPT-5 mini new run'),
    ('anthropic', 'claude-sonnet-4-5'): ('new_anthropic_claude_sonnet_4_5', 'Claude Sonnet 4.5'),
}

labels_long['model_key'] = labels_long.apply(lambda r: model_key_map[(r['provider'], r['model'])][0], axis=1)
labels_long['model_name'] = labels_long.apply(lambda r: model_key_map[(r['provider'], r['model'])][1], axis=1)
labels_long['category'] = labels_long['category'].apply(normalize_label)
labels_long['evidence'] = labels_long['evidence'].fillna('').astype(str).str.strip()

expected_counts = labels_long.groupby(['provider', 'model', 'run_status']).size().reset_index(name='rows')
display(expected_counts)

if labels_long.groupby(['provider', 'model'])['item_id'].nunique().min() < 100:
    print('Warning: At least one model has fewer than 100 ok labels. Metrics will use available rows only.')

,provider,model,run_status,rows
0,anthropic,claude-sonnet-4-5,ok,100
1,openai,gpt-5-mini,ok,100
2,openai,gpt-5.4,ok,100
3,openai,gpt-5.4-mini,ok,100


In [4]:
base_cols = ['item_id', 'hit_id', 'row_id', 'outlet', 'article_title', 'entity_mention', 'text', 'gold_label']

old_rows = manual.loc[:, base_cols + ['old_gpt_5_mini_category', 'old_gpt_5_mini_evidence']].copy()
old_rows = old_rows.rename(columns={
    'old_gpt_5_mini_category': 'predicted_label',
    'old_gpt_5_mini_evidence': 'evidence',
})
old_rows['provider'] = 'manual_csv'
old_rows['model'] = 'old_gpt_5_mini'
old_rows['model_key'] = 'old_gpt_5_mini'
old_rows['model_name'] = 'Old GPT-5 mini label'
old_rows['run_status'] = np.where(old_rows['predicted_label'].notna(), 'ok', 'missing')
old_rows['error'] = ''

new_rows = manual.loc[:, base_cols].merge(
    labels_long.loc[:, ['item_id', 'provider', 'model', 'model_key', 'model_name', 'category', 'evidence', 'run_status', 'error']],
    on='item_id',
    how='left',
)
new_rows = new_rows.rename(columns={'category': 'predicted_label'})

_eval_cols = [
    'item_id', 'hit_id', 'row_id', 'outlet', 'article_title', 'entity_mention', 'text',
    'gold_label', 'provider', 'model', 'model_key', 'model_name',
    'predicted_label', 'evidence', 'run_status', 'error'
]
eval_long = pd.concat([old_rows.loc[:, _eval_cols], new_rows.loc[:, _eval_cols]], ignore_index=True)
eval_long['gold_available'] = eval_long['gold_label'].notna()
eval_long['prediction_available'] = eval_long['predicted_label'].notna() & eval_long['run_status'].eq('ok')
eval_long['correct'] = pd.Series(pd.NA, index=eval_long.index, dtype='boolean')
_score_mask = eval_long['gold_available'] & eval_long['prediction_available']
eval_long.loc[_score_mask, 'correct'] = (
    eval_long.loc[_score_mask, 'gold_label'].astype(str).to_numpy()
    == eval_long.loc[_score_mask, 'predicted_label'].astype(str).to_numpy()
)

eval_long.to_csv(EVAL_LONG_PATH, index=False)
print('Evaluation long rows:', len(eval_long))
print('Saved:', EVAL_LONG_PATH)
display(eval_long.head(15))

Evaluation long rows: 500
Saved: /Users/MattisHaumann/Dev/Thesis/03_Framing/outputs/test_runs/framing_gold_model_evaluation_long.csv


,item_id,hit_id,row_id,outlet,article_title,entity_mention,text,gold_label,provider,model,model_key,model_name,predicted_label,evidence,run_status,error,gold_available,prediction_available,correct
0,1,5718ec8e2e0e1a02,2722,Nius,"„Rechtsextrem“, „rassistisch“, „Fake News“: Diese Lügen verbreiten deutsche Medien unmittelbar nach Kirks Ermordung über ihn",Spiegel | Tagesschau,"Das Attentat auf Charlie Kirk rührte viele Menschen wie hier in Utah – linke deutsche Medien sahen dagegen Anlass für Märchenerzählungen. Der einzige Vorwurf, den der Spiegel ü...",DISINFORMATION/FALSCHDARSTELLUNG,manual_csv,old_gpt_5_mini,old_gpt_5_mini,Old GPT-5 mini label,POSITIONS-/PARTEILICHKEITS-BIAS,linke deutsche Medien,ok,,True,True,False
1,2,7887b57e328ce932,4764,Nius,"Queere Tiere auf der Arche, Windel-Hühnchen in der Kirche: So verrückt war 2025",ZDF | Stern,"Merz, wieder einmal wahrheitswidrig: „Es gibt zwischen der CDU und der AfD keine Gemeinsamkeiten.“ Eine Schlagzeile, wie sie nur in Deutschland denkbar ist: „Unbekannte hissen ...",NaN,manual_csv,old_gpt_5_mini,old_gpt_5_mini,Old GPT-5 mini label,VERZERRUNG/MANIPULATION,Rechte Klassenzimmer,ok,,False,True,<NA>
2,3,168f480d060f709d,4920,Nius,Der Winter hat seine Unschuld verloren: Früher bedeutete Schnee pure Lebensfreude – heute wird den Kindern Angst gemacht,Tagesschau,Zudem: Die angekündigten extremen Schneefälle ließen am Ende auf sich warten. Betreutes Laufen: Diese Anleitung postete die Tagesschau auf Instagram. Die Natur wird zusehends z...,NEUTRAL,manual_csv,old_gpt_5_mini,old_gpt_5_mini,Old GPT-5 mini label,VERZERRUNG/MANIPULATION,zu unserem Feind stilisiert,ok,,True,True,False
3,4,b3ebf0fb7f6da9a9,9073,RT_de,"Lanz in Panik: ""Russland hat sich nach Westen ausgeweitet""",Markus Lanz | ZDF,"Lanz in Panik: ""Russland hat sich nach Westen ausgeweitet"". Von Alexej Danckwardt Wussten Sie, dass nicht die NATO sich nach 1991 um Hunderte Kilometer nach Osten ausgedehnt ha...",VERSAGEN/INKOMPETENZ,manual_csv,old_gpt_5_mini,old_gpt_5_mini,Old GPT-5 mini label,VERSAGEN/INKOMPETENZ,weniger Gegenwehr entgegenzusetzen,ok,,True,True,True
4,5,df33ae235b273262,7663,RT_de,"ZDF-Mitarbeiter beklagen interne Repressionen und fehlende ""innere Pressefreiheit""",WDR | ZDF,"Tucker Carlson: Zensur schützt die Mächtigen, nicht die Schwachen Im Verlauf der Anhörung benannte Halbach zwei Beispiele für ""Einschüchterungsversuche"" gegenüber kritischen St...",VERZERRUNG/MANIPULATION,manual_csv,old_gpt_5_mini,old_gpt_5_mini,Old GPT-5 mini label,POSITIONS-/PARTEILICHKEITS-BIAS,"Zensur schützt die Mächtigen, nicht die Schwachen",ok,,True,True,False
5,6,4714abd36db0c52e,3438,Nius,Gefangen im Land der Durchhalteparolen: Das hohle Stadtbild-Gedröhne des Friedrich Merz,Tagesschau,"Das Stadtbild ändert sich, ohne dass die Bevölkerung jemals gefragt wurde, ob es diese Veränderung wirklich in Kauf nehmen will. ## Synthese mit einem expansiven Islam Der vere...",NEUTRAL,manual_csv,old_gpt_5_mini,old_gpt_5_mini,Old GPT-5 mini label,POSITIONS-/PARTEILICHKEITS-BIAS,gebilligt von Tagesschau-Boomern,ok,,True,True,False
6,7,a8324d32b15f68db,11397,Tichys_Einblick,"Wacht auf, ihr Schlafwandler!",NDR,"Wacht auf, ihr Schlafwandler! Im literarischen Freiheitschor gegen deutsche Staatsgläubigkeit dominieren wenige markante Stimmen: So geben beispielsweise der Buchautor Gerald G...",NEUTRAL,manual_csv,old_gpt_5_mini,old_gpt_5_mini,Old GPT-5 mini label,NEUTRAL,,ok,,True,True,True
7,8,c81d317f74eb603d,425,Antispiegel,New York Times: „Selenskys Regierung sabotierte die Aufsicht und ließ Korruption gedeihen“,Spiegel | Qualitätsmedien,"Selensky habe damit rein gar nichts zu tun, auch wenn sein gesamtes Umfeld darin verstrickt ist und das NABU offen davon spricht, sie hätten den Schutz und die Nähe Selenskys f...",VERZERRUNG/MANIPULATION,manual_csv,old_gpt_5_mini,old_gpt_5_mini,Old GPT-5 mini label,VERZERRUNG/MANIPULATION,Spiegel-Propagandist,ok,,True,True,True
8,9,c948446bc708c8ff,3289,Nius,"An alle, die Trump mit Hitler verglichen haben –

In [5]:
def cohen_kappa(y_true, y_pred):
    pairs = pd.DataFrame({'true': y_true, 'pred': y_pred}).dropna()
    if pairs.empty:
        return np.nan
    labels = sorted(set(pairs['true']).union(set(pairs['pred'])))
    matrix = pd.crosstab(pairs['true'], pairs['pred']).reindex(index=labels, columns=labels, fill_value=0)
    n = matrix.to_numpy().sum()
    if n == 0:
        return np.nan
    observed = np.trace(matrix.to_numpy()) / n
    row_marginals = matrix.sum(axis=1).to_numpy()
    col_marginals = matrix.sum(axis=0).to_numpy()
    expected = (row_marginals @ col_marginals) / (n ** 2)
    if math.isclose(1 - expected, 0):
        return 1.0 if math.isclose(observed, 1.0) else np.nan
    return (observed - expected) / (1 - expected)

scored = eval_long.loc[eval_long['gold_available'] & eval_long['prediction_available']].copy()

def summarize_group(group):
    return pd.Series({
        'n_gold_scored': len(group),
        'correct': int(group['correct'].sum()),
        'accuracy': float(group['correct'].mean()) if len(group) else np.nan,
        'kappa': cohen_kappa(group['gold_label'], group['predicted_label']),
    })

overall_summary = (
    scored.groupby(['model_key', 'model_name'], sort=False)
    .apply(summarize_group, include_groups=False)
    .reset_index()
    .sort_values('accuracy', ascending=False)
)
overall_summary['accuracy_pct'] = (overall_summary['accuracy'] * 100).round(1)
overall_summary['kappa'] = overall_summary['kappa'].round(3)

overall_summary.to_csv(KAPPA_SUMMARY_PATH, index=False)
print('Saved:', KAPPA_SUMMARY_PATH)
display(overall_summary)

Saved: /Users/MattisHaumann/Dev/Thesis/03_Framing/outputs/test_runs/framing_gold_kappa_summary.csv


,model_key,model_name,n_gold_scored,correct,accuracy,kappa,accuracy_pct
2,new_openai_gpt_5_mini,GPT-5 mini new run,98.0,66.0,0.673469,0.590,67.3
4,new_openai_gpt_5_4_mini,GPT-5.4 mini,98.0,66.0,0.673469,0.580,67.3
0,old_gpt_5_mini,Old GPT-5 mini label,98.0,63.0,0.642857,0.554,64.3
3,new_openai_gpt_5_4,GPT-5.4,98.0,62.0,0.632653,0.540,63.3
1,new_anthropic_claude_sonnet_4_5,Claude Sonnet 4.5,98.0,60.0,0.612245,0.508,61.2


In [6]:
outlet_summary = (
    scored.groupby(['outlet', 'model_key', 'model_name'], sort=False)
    .apply(summarize_group, include_groups=False)
    .reset_index()
)
outlet_summary['accuracy_pct'] = (outlet_summary['accuracy'] * 100).round(1)
outlet_summary['kappa'] = outlet_summary['kappa'].round(3)
outlet_summary.to_csv(OUTLET_ACCURACY_PATH, index=False)

accuracy_pivot = outlet_summary.pivot_table(
    index='outlet',
    columns='model_name',
    values='accuracy_pct',
    aggfunc='first',
).reset_index()
accuracy_pivot.to_csv(OUTLET_ACCURACY_PIVOT_PATH, index=False)

kappa_pivot = outlet_summary.pivot_table(
    index='outlet',
    columns='model_name',
    values='kappa',
    aggfunc='first',
).reset_index()
outlet_summary.to_csv(OUTLET_KAPPA_PATH, index=False)

print('Accuracy by outlet saved:', OUTLET_ACCURACY_PATH)
print('Accuracy pivot saved:', OUTLET_ACCURACY_PIVOT_PATH)
print('Outlet kappa saved:', OUTLET_KAPPA_PATH)
print('Accuracy % by outlet:')
display(accuracy_pivot)
print('Kappa by outlet:')
display(kappa_pivot)

Accuracy by outlet saved: /Users/MattisHaumann/Dev/Thesis/03_Framing/outputs/test_runs/framing_gold_accuracy_by_outlet.csv
Accuracy pivot saved: /Users/MattisHaumann/Dev/Thesis/03_Framing/outputs/test_runs/framing_gold_accuracy_by_outlet_pivot.csv
Outlet kappa saved: /Users/MattisHaumann/Dev/Thesis/03_Framing/outputs/test_runs/framing_gold_kappa_by_outlet.csv
Accuracy % by outlet:


model_name,outlet,Claude Sonnet 4.5,GPT-5 mini new run,GPT-5.4,GPT-5.4 mini,Old GPT-5 mini label
0,Antispiegel,75.0,75.0,50.0,75.0,75.0
1,Compact,80.0,80.0,80.0,80.0,80.0
2,Deutschlandkurier,80.0,80.0,80.0,60.0,80.0
3,Nius,59.1,63.6,63.6,63.6,54.5
4,RT_de,63.6,90.9,72.7,81.8,72.7
5,Tagesschau,75.0,75.0,66.7,83.3,66.7
6,Tichys_Einblick,48.6,54.3,57.1,57.1,60.0


Kappa by outlet:


model_name,outlet,Claude Sonnet 4.5,GPT-5 mini new run,GPT-5.4,GPT-5.4 mini,Old GPT-5 mini label
0,Antispiegel,0.652,0.667,0.347,0.652,0.673
1,Compact,0.722,0.722,0.722,0.667,0.722
2,Deutschlandkurier,0.737,0.750,0.750,0.545,0.750
3,Nius,0.479,0.548,0.552,0.526,0.442
4,RT_de,0.527,0.880,0.653,0.744,0.649
5,Tagesschau,-0.091,0.321,-0.091,0.442,0.127
6,Tichys_Einblick,0.367,0.409,0.455,0.459,0.493


In [7]:
# Wide comparison file: one row per item, with gold + old GPT + all new model labels/evidence.
wide = manual.loc[:, ['item_id', 'hit_id', 'row_id', 'outlet', 'article_title', 'entity_mention', 'text', 'gold_label']].copy()

model_order = [
    ('old_gpt_5_mini', 'old_gpt_5_mini'),
    ('new_openai_gpt_5_4', 'gpt_5_4'),
    ('new_openai_gpt_5_4_mini', 'gpt_5_4_mini'),
    ('new_openai_gpt_5_mini', 'gpt_5_mini_new'),
    ('new_anthropic_claude_sonnet_4_5', 'claude_sonnet_4_5'),
]

for model_key, prefix in model_order:
    part = eval_long.loc[eval_long['model_key'].eq(model_key), [
        'item_id', 'predicted_label', 'evidence', 'run_status', 'error', 'correct'
    ]].copy()
    part = part.rename(columns={
        'predicted_label': f'{prefix}_category',
        'evidence': f'{prefix}_evidence',
        'run_status': f'{prefix}_status',
        'error': f'{prefix}_error',
        'correct': f'{prefix}_correct',
    })
    wide = wide.merge(part, on='item_id', how='left')

wide.to_csv(EVAL_WIDE_PATH, index=False)
print('Saved:', EVAL_WIDE_PATH)
print('Wide shape:', wide.shape)
display(wide.head(10))

Saved: /Users/MattisHaumann/Dev/Thesis/03_Framing/outputs/test_runs/framing_gold_model_evaluation_wide.csv
Wide shape: (100, 33)


,item_id,hit_id,row_id,outlet,article_title,entity_mention,text,gold_label,old_gpt_5_mini_category,old_gpt_5_mini_evidence,old_gpt_5_mini_status,old_gpt_5_mini_error,old_gpt_5_mini_correct,gpt_5_4_category,gpt_5_4_evidence,gpt_5_4_status,gpt_5_4_error,gpt_5_4_correct,gpt_5_4_mini_category,gpt_5_4_mini_evidence,gpt_5_4_mini_status,gpt_5_4_mini_error,gpt_5_4_mini_correct,gpt_5_mini_new_category,gpt_5_mini_new_evidence,gpt_5_mini_new_status,gpt_5_mini_new_error,gpt_5_mini_new_correct,claude_sonnet_4_5_category,claude_sonnet_4_5_evidence,claude_sonnet_4_5_status,claude_sonnet_4_5_error,claude_sonnet_4_5_correct
0,1,5718ec8e2e0e1a02,2722,Nius,"„Rechtsextrem“, „rassistisch“, „Fake News“: Diese Lügen verbreiten deutsche Medien unmittelbar nach Kirks Ermordung über ihn",Spiegel | Tagesschau,"Das Attentat auf Charlie Kirk rührte viele Menschen wie hier in Utah – linke deutsche Medien sahen dagegen Anlass für Märchenerzählungen. Der einzige Vorwurf, den der Spiegel ü...",DISINFORMATION/FALSCHDARSTELLUNG,POSITIONS-/PARTEILICHKEITS-BIAS,linke deutsche Medien,ok,,False,DISINFORMATION/FALSCHDARSTELLUNG,linke deutsche Medien sahen dagegen Anlass für Märchenerzählungen,ok,NaN,True,VERZERRUNG/MANIPULATION,den liberalen Hintergrund und die Idee des amerikanischen Waffenrechts aber lässt der Spiegel unerwähnt,ok,NaN,False,POSITIONS-/PARTEILICHKEITS-BIAS,linke deutsche Medien sahen dagegen Anlass für Märchenerzählungen,ok,NaN,False,VERZERRUNG/MANIPULATION,"lässt der Spiegel unerwähnt. Man kann davon ausgehen, dass er bei nicht-westlichen Kulturen solche kulturellen Hintergründe erwähnt hätte",ok,NaN,False
1,2,7887b57e328ce932,4764,Nius,"Queere Tiere auf der Arche, Windel-Hühnchen in der Kirche: So verrückt war 2025",ZDF | Stern,"Merz, wieder einmal wahrheitswidrig: „Es gibt zwischen der CDU und der AfD keine Gemeinsamkeiten.“ Eine Schlagzeile, wie sie nur in Deutschland denkbar ist: „Unbekannte hissen ...",NaN,VERZERRUNG/MANIPULATION,Rechte Klassenzimmer,ok,,<NA>,DISINFORMATION/FALSCHDARSTELLUNG,"Israel bezichtigt, gezielt einen unschuldigen Pressemenschen umgebracht zu haben. Dann stellt sich heraus, dass er Mitglied der Hamas war",ok,NaN,<NA>,DISINFORMATION/FALSCHDARSTELLUNG,"bezichtigt, Israel ... gezielt einen unschuldigen Pressemenschen umgebracht zu haben. Dann stellt sich heraus, dass er Mitglied der Hamas war. Dumm gelaufen, ZDF!",ok,NaN,<NA>,VERSAGEN/INKOMPETENZ,"Dumm gelaufen, ZDF!",ok,NaN,<NA>,DISINFORMATION/FALSCHDARSTELLUNG,"Ein von Israel neutralisierter Mitarbeiter einer Partnerfirma des ZDF wird vom Sender betrauert, Israel bezichtigt, gezielt einen unschuldigen Pressemenschen umgebracht zu habe...",ok,NaN,<NA>
2,3,168f480d060f709d,4920,Nius,Der Winter hat seine Unschuld verloren: Früher bedeutete Schnee pure Lebensfreude – heute wird den Kindern Angst gemacht,Tagesschau,Zudem: Die angekündigten extremen Schneefälle ließen am Ende auf sich warten. Betreutes Laufen: Diese Anleitung postete die Tagesschau auf Instagram. Die Natur wird zusehends z...,NEUTRAL,VERZERRUNG/MANIPULATION,zu unserem Feind stilisiert,ok,,False,VERZERRUNG/MANIPULATION,Die Natur wird zusehends zu unserem Feind stilisiert,ok,NaN,False,VERZERRUNG/MANIPULATION,Diese Anleitung postete die Tagesschau auf Instagram,ok,NaN,False,VERZERRUNG/MANIPULATION,Die Natur wird zusehends zu unserem Feind stilisiert,ok,NaN,False,VERZERRUNG/MANIPULATION,Betreutes Laufen: Diese Anleitung postete die Tagesschau auf Instagram. Die Natur wird zusehends zu unserem Feind stilisiert,ok,NaN,False
3,4,b3ebf0fb7f6da9a9,9073,RT_de,"Lanz in Panik: ""Russland hat sich nach Westen ausgeweitet""",Markus Lanz | ZDF,"Lanz in Panik: ""Russland hat sich nach Westen ausgeweitet"". Von Alexej Danckwardt Wussten Sie, dass nicht die NATO sich nach 1991 um Hunderte Kilometer nach Osten ausgedehnt ha...",VERSAGEN/INKOMPETENZ,VERSAGEN/INKOMPETENZ,weniger Gegenwehr entgegenzusetzen,ok,,True,VERSAGEN/INKOMPETENZ,das Abstruseste im Podcast,ok,NaN,True,NEUTRAL,,ok,NaN,False,VERSAGEN/INKOMPETENZ,"Precht vermag

In [8]:
# Disagreement inspection: where models differ from GOLD TRUTH.
errors_for_review = (
    eval_long.loc[eval_long['gold_available'] & eval_long['prediction_available'] & eval_long['correct'].eq(False)]
    .sort_values(['item_id', 'model_name'])
    .loc[:, ['item_id', 'row_id', 'outlet', 'entity_mention', 'gold_label', 'model_name', 'predicted_label', 'evidence']]
)
print('Model-vs-gold disagreements:', len(errors_for_review))
display(errors_for_review.head(50))


Model-vs-gold disagreements: 173


,item_id,row_id,outlet,entity_mention,gold_label,model_name,predicted_label,evidence
100,1,2722,Nius,Spiegel | Tagesschau,DISINFORMATION/FALSCHDARSTELLUNG,Claude Sonnet 4.5,VERZERRUNG/MANIPULATION,"lässt der Spiegel unerwähnt. Man kann davon ausgehen, dass er bei nicht-westlichen Kulturen solche kulturellen Hintergründe erwähnt hätte"
101,1,2722,Nius,Spiegel | Tagesschau,DISINFORMATION/FALSCHDARSTELLUNG,GPT-5 mini new run,POSITIONS-/PARTEILICHKEITS-BIAS,linke deutsche Medien sahen dagegen Anlass für Märchenerzählungen
103,1,2722,Nius,Spiegel | Tagesschau,DISINFORMATION/FALSCHDARSTELLUNG,GPT-5.4 mini,VERZERRUNG/MANIPULATION,den liberalen Hintergrund und die Idee des amerikanischen Waffenrechts aber lässt der Spiegel unerwähnt
0,1,2722,Nius,Spiegel | Tagesschau,DISINFORMATION/FALSCHDARSTELLUNG,Old GPT-5 mini label,POSITIONS-/PARTEILICHKEITS-BIAS,linke deutsche Medien
108,3,4920,Nius,Tagesschau,NEUTRAL,Claude Sonnet 4.5,VERZERRUNG/MANIPULATION,Betreutes Laufen: Diese Anleitung postete die Tagesschau auf Instagram. Die Natur wird zusehends zu unserem Feind stilisiert
109,3,4920,Nius,Tagesschau,NEUTRAL,GPT-5 mini new run,VERZERRUNG/MANIPULATION,Die Natur wird zusehends zu unserem Feind stilisiert
110,3,4920,Nius,Tagesschau,NEUTRAL,GPT-5.4,VERZERRUNG/MANIPULATION,Die Natur wird zusehends zu unserem Feind stilisiert
111,3,4920,Nius,Tagesschau,NEUTRAL,GPT-5.4 mini,VERZERRUNG/MANIPULATION,Diese Anleitung postete die Tagesschau auf Instagram
2,3,4920,Nius,Tagesschau,NEUTRAL,Old GPT-5 mini label,VERZERRUNG/MANIPULATION,zu unserem Feind stilisiert
112,4,9073,RT_de,Markus Lanz | ZDF,VERSAGEN/INKOMPETENZ,Claude Sonnet 4.5,VERZERRUNG/MANIPULATION,"kaum etwas anderes dokumentiert den Tod und die inzwischen weit vorangeschrittene Verwesung alles Intellektuellen in Deutschland besser als der nekrophile Porno, den die beiden..."


## Model Significance Test

This table tests each model against the selected reference model (`GPT-5 mini new run`). Because every model labels the same gold items, the relevant test is a paired exact McNemar test on row-level `correct`/`incorrect` outcomes. The null hypothesis is that both models have the same error rate on the gold sample.


In [ ]:
# Exact McNemar significance table: each model vs the selected reference model.
# Basis: same 98 gold-labelled items, compared as paired correct/incorrect outcomes.

import math

baseline_model = "GPT-5 mini new run"
significance_path = OUTPUT_DIR / "framing_gold_model_significance_vs_gpt5mini.csv"

scored_for_sig = eval_long[
    eval_long["gold_available"].eq(True) & eval_long["prediction_available"].eq(True)
].copy()
scored_for_sig["correct_bool"] = scored_for_sig["correct"].astype(str).str.lower().eq("true")
wide_correct = scored_for_sig.pivot(index="item_id", columns="model_name", values="correct_bool")

def wilson_ci(k, n, z=1.96):
    if n == 0:
        return float("nan"), float("nan")
    p = k / n
    denom = 1 + z * z / n
    center = (p + z * z / (2 * n)) / denom
    margin = z * math.sqrt((p * (1 - p) + z * z / (4 * n)) / n) / denom
    return center - margin, center + margin

def exact_mcnemar_p(b, c):
    # b = baseline correct and comparison model wrong
    # c = baseline wrong and comparison model correct
    n = b + c
    if n == 0:
        return 1.0
    smaller = min(b, c)
    p = 2 * sum(math.comb(n, i) * (0.5 ** n) for i in range(smaller + 1))
    return min(1.0, p)

sig_rows = []
for _, row in kappa_summary.iterrows():
    model_name = row["model_name"]
    n = int(row["n_gold_scored"])
    correct = int(row["correct"])
    acc = correct / n
    ci_low, ci_high = wilson_ci(correct, n)

    if model_name == baseline_model:
        b = c = None
        p_value = None
        tested_against = "reference model"
        interpretation = "Reference: no pairwise test against itself."
    else:
        pair = wide_correct[[baseline_model, model_name]].dropna()
        b = int(((pair[baseline_model] == True) & (pair[model_name] == False)).sum())
        c = int(((pair[baseline_model] == False) & (pair[model_name] == True)).sum())
        p_value = exact_mcnemar_p(b, c)
        tested_against = baseline_model
        interpretation = "significant difference" if p_value < 0.05 else "not significant"

    sig_rows.append({
        "model_name": model_name,
        "n_gold_scored": n,
        "correct": correct,
        "accuracy_pct": round(acc * 100, 1),
        "accuracy_95ci": f"{ci_low * 100:.1f}-{ci_high * 100:.1f}",
        "kappa": round(float(row["kappa"]), 3),
        "test": "Exact McNemar",
        "tested_against": tested_against,
        "baseline_correct_model_wrong": b,
        "baseline_wrong_model_correct": c,
        "p_value_vs_gpt5mini": None if p_value is None else round(p_value, 3),
        "significant_at_0_05": "" if p_value is None else ("yes" if p_value < 0.05 else "no"),
        "interpretation": interpretation,
    })

significance_table = (
    pd.DataFrame(sig_rows)
    .sort_values(["accuracy_pct", "kappa"], ascending=[False, False])
    .reset_index(drop=True)
)

significance_table.to_csv(significance_path, index=False)
display(significance_table)
print("Saved:", significance_path)
